In [2]:
# ============================================================
# 01_Python_gpu_biogeme.ipynb
# Biogeme GPU benchmark for the Apollo Swiss route-choice MXL model
#
# This notebook estimates the Apollo Swiss route-choice mixed
# logit specification with Biogeme using JAX on an NVIDIA GPU.
#
# Target environment:
# - Google Colab.
# - NVIDIA Tesla T4.
# - CUDA 12 JAX backend.
#
# Scope:
# - Biogeme BFGS estimation.
# - Package-native simulated draws.
# - One fresh Python worker per repetition to avoid memory carry-over.
# - Repeated runs summarised by draw size to measure runtime variation.
# - GPU, JAX, package, and environment records for reproducibility.
# - Estimation time taken from Biogeme's reported Optimization time.
#
# Main outputs:
# - results/raw/biogeme_gpu_*.json
# - results/raw/biogeme_gpu_rep_results.csv
# - results/benchmark_summary_biogeme_gpu.csv
# - results/session_info_biogeme_gpu.txt
# - results/biogeme_gpu_results.zip
# ============================================================


In [3]:
# ============================================================
# 1. Biogeme 3.3.4 GPU benchmark environment
# ============================================================

# ------------------------------------------------------------
# 1.1 Enable Python 3.12 virtual environment support
# ------------------------------------------------------------

!apt-get update -qq
!apt-get install -y -qq python3.12-venv


# ------------------------------------------------------------
# 1.2 Create clean virtual environment
# ------------------------------------------------------------

import shutil
import subprocess
from pathlib import Path

ENV = Path("/content/biogeme334")
PYTHON = ENV / "bin" / "python"

if ENV.exists():
    shutil.rmtree(ENV)

subprocess.run(
    ["/usr/bin/python3.12", "-m", "venv", str(ENV)],
    check=True
)

print("Virtual environment:", PYTHON)


# ------------------------------------------------------------
# 1.3 Upgrade pip
# ------------------------------------------------------------

subprocess.run(
    [str(PYTHON), "-m", "pip", "install", "--upgrade", "pip"],
    check=True
)


# ------------------------------------------------------------
# 1.4 Install numerical dependencies and CUDA-enabled JAX
# ------------------------------------------------------------

subprocess.run(
    [
        str(PYTHON), "-m", "pip", "install",
        "numpy==2.4.6",
        "scipy==1.18.0",
        "pandas==2.3.3",
        "jax[cuda12]==0.11.0",
        "biogeme_optimization==0.0.11",
    ],
    check=True
)


# ------------------------------------------------------------
# 1.5 Install Biogeme
# ------------------------------------------------------------

subprocess.run(
    [
        str(PYTHON), "-m", "pip", "install",
        "biogeme==3.3.4",
    ],
    check=True
)

print("Biogeme 3.3.4 installation completed.")


# ------------------------------------------------------------
# 1.6 Verify environment and record GPU information
# ------------------------------------------------------------

check_code = r'''
import json
import platform
import re
import subprocess
from importlib.metadata import PackageNotFoundError, version

import numpy as np
import scipy
import pandas as pd
import jax

from biogeme.database import Database


def cmd(command):
    return subprocess.check_output(
        command,
        shell=True,
        text=True
    ).strip()


def safe_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return "not installed"


gpu_line = cmd(
    "nvidia-smi --query-gpu=name,memory.total,driver_version "
    "--format=csv,noheader,nounits | head -1"
)
gpu_name, gpu_memory_mb, driver_version = [
    x.strip() for x in gpu_line.split(",", 2)
]

nvidia_smi_output = cmd("nvidia-smi")
cuda_match = re.search(r"CUDA Version:\s*([0-9.]+)", nvidia_smi_output)
cuda_reported = cuda_match.group(1) if cuda_match else "unknown"

info = {
    "Python": platform.python_version(),
    "Biogeme": version("biogeme"),
    "biogeme_optimization": version("biogeme_optimization"),
    "NumPy": np.__version__,
    "SciPy": scipy.__version__,
    "Pandas": pd.__version__,
    "JAX": jax.__version__,
    "JAXLIB": version("jaxlib"),
    "JAX_CUDA12_plugin": safe_version("jax-cuda12-plugin"),
    "JAX_CUDA12_PJRT": safe_version("jax-cuda12-pjrt"),
    "JAX_backend": jax.default_backend(),
    "JAX_devices": str(jax.devices()),
    "GPU_model": gpu_name,
    "GPU_memory_MB": gpu_memory_mb,
    "NVIDIA_driver": driver_version,
    "CUDA_version_reported_by_nvidia_smi": cuda_reported,
}

print("=" * 60)
print("Biogeme GPU benchmark environment")
print("=" * 60)

for key, value in info.items():
    print(f"{key:<36}: {value}")

print("\nBiogeme Database import: SUCCESS")

with open("/content/biogeme334_environment.json", "w") as f:
    json.dump(info, f, indent=2)
'''

result = subprocess.run(
    [str(PYTHON), "-c", check_code],
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package python3.12-venv.
(Reading database ... 122492 files and directories currently installed.)
Preparing to unpack .../python3.12-venv_3.12.13-1+jammy1_amd64.deb ...
Unpacking python3.12-venv (3.12.13-1+jammy1) ...
Setting up python3.12-venv (3.12.13-1+jammy1) ...
Virtual environment: /content/biogeme334/bin/python
Biogeme 3.3.4 installation completed.
Biogeme GPU benchmark environment
Python                              : 3.12.13
Biogeme                             : 3.3.4
biogeme_optimization                : 0.0.11
NumPy                               : 2.4.6
SciPy                               : 1.18.0
Pandas                              : 2.3.3
JAX                                 : 0.11.0
JAXLIB                              : 0.11.0
JAX_CUDA12_plugin            

In [4]:
# ============================================================
# 2. Benchmark settings and data
# ============================================================

from pathlib import Path
import subprocess, json
import pandas as pd

proj_root = Path.cwd()

raw_results_dir = proj_root / "results" / "raw"
summary_file = proj_root / "results" / "benchmark_summary_biogeme_gpu.csv"
session_info_file = proj_root / "results" / "session_info_biogeme_gpu.txt"

raw_results_dir.mkdir(parents=True, exist_ok=True)

draws_grid = [500, 1000, 2000, 5000, 10000]
seed = 123
n_reps = 5

# Load data
data_file = proj_root / "data" / "apollo_swiss_route_choice.csv"
url = "https://raw.githubusercontent.com/douxin1060948912-ai/Anvl/main/apollo_swiss_route_choice.csv"

if data_file.exists():
    df = pd.read_csv(data_file)
else:
    data_file.parent.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(url)
    df.to_csv(data_file, index=False)

df = df.sort_values("ID").reset_index(drop=True)
csv_path = str(data_file)

# Get actual Biogeme/JAX/GPU environment information
code = r'''
import json
import platform
import re
import subprocess
from importlib.metadata import PackageNotFoundError, version

import jax


def safe_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return "not installed"


gpu_line = subprocess.check_output(
    "nvidia-smi --query-gpu=name,memory.total,driver_version "
    "--format=csv,noheader,nounits | head -1",
    shell=True,
    text=True,
).strip()
gpu_name, gpu_memory_mb, driver_version = [
    x.strip() for x in gpu_line.split(",", 2)
]

nvidia_smi_output = subprocess.check_output(
    "nvidia-smi",
    shell=True,
    text=True,
).strip()
cuda_match = re.search(r"CUDA Version:\s*([0-9.]+)", nvidia_smi_output)
cuda_reported = cuda_match.group(1) if cuda_match else "unknown"

print(json.dumps({
    "Python": platform.python_version(),
    "Biogeme": version("biogeme"),
    "JAX": version("jax"),
    "JAXLIB": version("jaxlib"),
    "JAX_CUDA12_plugin": safe_version("jax-cuda12-plugin"),
    "JAX_CUDA12_PJRT": safe_version("jax-cuda12-pjrt"),
    "NumPy": version("numpy"),
    "SciPy": version("scipy"),
    "Pandas": version("pandas"),
    "Backend": jax.default_backend(),
    "Devices": str(jax.devices()),
    "GPU_model": gpu_name,
    "GPU_memory_MB": gpu_memory_mb,
    "NVIDIA_driver": driver_version,
    "CUDA_version_reported_by_nvidia_smi": cuda_reported,
}))
'''

env_info = json.loads(
    subprocess.check_output(
        [str(PYTHON), "-c", code],
        text=True
    )
)

session_info_file.write_text(
    "\n".join(f"{k}: {v}" for k, v in env_info.items()),
    encoding="utf-8"
)

print("Draws:", draws_grid)
print("Seed:", seed)
print("Reps:", n_reps)
print("Individuals:", df["ID"].nunique())
print("Observations:", len(df))
print(pd.Series(env_info))


Draws: [500, 1000, 2000, 5000, 10000]
Seed: 123
Reps: 5
Individuals: 388
Observations: 3492
Python                                            3.12.13
Biogeme                                             3.3.4
JAX                                                0.11.0
JAXLIB                                             0.11.0
JAX_CUDA12_plugin                                  0.11.0
JAX_CUDA12_PJRT                                    0.11.0
NumPy                                               2.4.6
SciPy                                              1.18.0
Pandas                                              2.3.3
Backend                                               gpu
Devices                                [CudaDevice(id=0)]
GPU_model                                        Tesla T4
GPU_memory_MB                                       15360
NVIDIA_driver                                   580.82.07
CUDA_version_reported_by_nvidia_smi                  13.0
dtype: object


In [5]:
%%writefile biogeme_worker.py
# ============================================================
# 3. Biogeme model definition + single-rep GPU runner
# ============================================================

import jax
import pandas as pd

import biogeme.biogeme_logging as blog
from biogeme.biogeme import BIOGEME
from biogeme.database import Database
from biogeme.expressions import (
    Beta,
    Draws,
    MonteCarlo,
    PanelLikelihoodTrajectory,
    Variable,
    exp,
    log,
)
from biogeme.models import logit


def run_one_rep(num_draws, seed, rep_id, csv_path):
    blog.get_screen_logger(level=blog.INFO)

    df = pd.read_csv(csv_path)
    database = Database("apollo_swiss_route_choice", df)
    database.panel("ID")

    CHOICE = Variable("choice")

    TT1 = Variable("tt1")
    TC1 = Variable("tc1")
    HW1 = Variable("hw1")
    CH1 = Variable("ch1")

    TT2 = Variable("tt2")
    TC2 = Variable("tc2")
    HW2 = Variable("hw2")
    CH2 = Variable("ch2")

    mu_log_b_tt    = Beta("mu_log_b_tt",    -3,   None, None, 0)
    sigma_log_b_tt = Beta("sigma_log_b_tt",  0.01, None, None, 0)
    mu_log_b_tc    = Beta("mu_log_b_tc",    -3,   None, None, 0)
    sigma_log_b_tc = Beta("sigma_log_b_tc",  0.01, None, None, 0)
    mu_log_b_hw    = Beta("mu_log_b_hw",    -3,   None, None, 0)
    sigma_log_b_hw = Beta("sigma_log_b_hw",  0.01, None, None, 0)
    mu_log_b_ch    = Beta("mu_log_b_ch",    -3,   None, None, 0)
    sigma_log_b_ch = Beta("sigma_log_b_ch",  0.01, None, None, 0)

    b_tt = -exp(mu_log_b_tt + sigma_log_b_tt * Draws("draws_tt", "NORMAL_MLHS_ANTI"))
    b_tc = -exp(mu_log_b_tc + sigma_log_b_tc * Draws("draws_tc", "NORMAL_MLHS_ANTI"))
    b_hw = -exp(mu_log_b_hw + sigma_log_b_hw * Draws("draws_hw", "NORMAL_MLHS_ANTI"))
    b_ch = -exp(mu_log_b_ch + sigma_log_b_ch * Draws("draws_ch", "NORMAL_MLHS_ANTI"))

    V1 = b_tt * TT1 + b_tc * TC1 + b_hw * HW1 + b_ch * CH1
    V2 = b_tt * TT2 + b_tc * TC2 + b_hw * HW2 + b_ch * CH2

    V = {1: V1, 2: V2}
    av = {1: 1, 2: 1}

    logprob = log(MonteCarlo(PanelLikelihoodTrajectory(logit(V, av, CHOICE))))

    bio = BIOGEME(
        database,
        logprob,
        number_of_draws=num_draws,
        seed=seed,
        generate_html=False,
        save_iterations=False,
        calculating_second_derivatives="never",
    )
    bio.model_name = f"biogeme_gpu_draws_{num_draws}"

    results = bio.estimate()

    # Use Biogeme's own reported optimization time as the estimation time.
    estimation_time = (
        results.optimization_messages["Optimization time"]
        .total_seconds()
    )

    return {
        "package": "Biogeme",
        "implementation": "biogeme_gpu",
        "backend": "GPU",
        "optimiser": "Biogeme_BFGS",
        "draws": int(num_draws),
        "seed": int(seed),
        "rep": int(rep_id),
        "jax_backend": jax.default_backend(),
        "jax_devices": "; ".join(str(device) for device in jax.devices()),
        "initial_loglik": float(results.raw_estimation_results.initial_log_likelihood),
        "final_loglik": float(results.final_log_likelihood),
        "estimation_time_sec": float(estimation_time),
        "optimization_messages": {
            key: str(value)
            for key, value in results.optimization_messages.items()
        },
    }


Writing biogeme_worker.py


In [6]:
# ============================================================
# 4. Run Biogeme GPU benchmark
# ============================================================

import time
import json
import subprocess
import numpy as np
import pandas as pd

rep_results = []
draw_results = []

gpu_info = {
    "gpu_model": subprocess.getoutput(
        "nvidia-smi --query-gpu=name --format=csv,noheader | head -1"
    ).strip(),
    "gpu_memory_mb": subprocess.getoutput(
        "nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -1"
    ).strip(),
    "nvidia_driver": subprocess.getoutput(
        "nvidia-smi --query-gpu=driver_version --format=csv,noheader | head -1"
    ).strip(),
}

for num_draws in draws_grid:

    print(f"\n{'=' * 60}")
    print(f"Biogeme GPU | draws = {num_draws}")
    print("=" * 60)

    times_this_draw = []
    ll_this_draw = []
    initial_loglik = None

    for rep_id in range(1, n_reps + 1):

        print(f"\nRep {rep_id} of {n_reps}")

        code = f'''
import json
from biogeme_worker import run_one_rep

r = run_one_rep(
    {num_draws},
    {seed},
    {rep_id},
    r'{csv_path}'
)

print("RESULT_JSON=" + json.dumps(r))
'''

        t0 = time.time()

        p = subprocess.run(
            [str(PYTHON), "-c", code],
            capture_output=True,
            text=True,
            check=True,
        )

        worker_wall_time = time.time() - t0

        result_line = [
            x for x in p.stdout.splitlines()
            if x.startswith("RESULT_JSON=")
        ][-1]

        result_record = json.loads(
            result_line.replace("RESULT_JSON=", "", 1)
        )

        result_record["worker_wall_time_sec"] = worker_wall_time
        result_record.update(gpu_info)

        if initial_loglik is None:
            initial_loglik = result_record["initial_loglik"]

        final_loglik = result_record["final_loglik"]
        estimation_time = result_record["estimation_time_sec"]

        result_file = (
            raw_results_dir /
            f"biogeme_gpu_draws_{num_draws}_rep_{rep_id}.json"
        )

        result_file.write_text(
            json.dumps(result_record, indent=2),
            encoding="utf-8",
        )

        rep_results.append({
            **result_record,
            "optimization_messages": None,
        })

        times_this_draw.append(estimation_time)
        ll_this_draw.append(final_loglik)

        print(f"Final log-likelihood: {final_loglik:.3f}")
        print(f"Estimation time     : {estimation_time:.3f} s")
        print(f"Worker wall time    : {worker_wall_time:.3f} s")

    times = np.array(times_this_draw)

    draw_results.append({
        "package": "Biogeme",
        "implementation": "biogeme_gpu",
        "backend": "GPU",
        "device": gpu_info["gpu_model"],
        "draws": num_draws,
        "seed": seed,
        "reps": n_reps,
        "initial_loglik": initial_loglik,
        "final_loglik": float(np.mean(ll_this_draw)),
        "mean_estimation_time_sec": float(times.mean()),
        "sd_estimation_time_sec": float(times.std(ddof=1)),
        "min_estimation_time_sec": float(times.min()),
        "max_estimation_time_sec": float(times.max()),
    })

    print(f"\nAverage estimation time: {times.mean():.4f} s")

rep_summary = pd.DataFrame(rep_results).drop(
    columns="optimization_messages"
)

rep_file = raw_results_dir / "biogeme_gpu_rep_results.csv"
rep_summary.to_csv(rep_file, index=False)

print(f"\nSaved to: {rep_file}")



Biogeme GPU | draws = 500

Rep 1 of 5
Final log-likelihood: -1447.294
Estimation time     : 11.955 s
Worker wall time    : 35.229 s

Rep 2 of 5
Final log-likelihood: -1447.294
Estimation time     : 11.876 s
Worker wall time    : 35.445 s

Rep 3 of 5
Final log-likelihood: -1447.294
Estimation time     : 11.886 s
Worker wall time    : 35.731 s

Rep 4 of 5
Final log-likelihood: -1447.294
Estimation time     : 12.173 s
Worker wall time    : 36.268 s

Rep 5 of 5
Final log-likelihood: -1447.294
Estimation time     : 12.146 s
Worker wall time    : 36.313 s

Average estimation time: 12.0071 s

Biogeme GPU | draws = 1000

Rep 1 of 5
Final log-likelihood: -1443.806
Estimation time     : 12.214 s
Worker wall time    : 37.065 s

Rep 2 of 5
Final log-likelihood: -1443.806
Estimation time     : 12.047 s
Worker wall time    : 36.560 s

Rep 3 of 5
Final log-likelihood: -1443.806
Estimation time     : 12.268 s
Worker wall time    : 35.959 s

Rep 4 of 5
Final log-likelihood: -1443.806
Estimation time  

In [7]:
# ============================================================
# 5. Summarise outputs
# ============================================================

benchmark_summary = pd.DataFrame(draw_results)

# Keep optimiser information
if "optimiser" not in benchmark_summary.columns:
    benchmark_summary["optimiser"] = "Biogeme_BFGS"

benchmark_summary = (
    benchmark_summary
    .sort_values(["draws", "optimiser"])
    .reset_index(drop=True)
)

# Save benchmark summary
benchmark_summary.to_csv(summary_file, index=False)

# Display summary
print(benchmark_summary)

print("\nSaved repetition-level results to:")
print(raw_results_dir / "biogeme_gpu_rep_results.csv")

print("\nSaved benchmark summary to:")
print(summary_file)

print("\nSaved session information to:")
print(session_info_file)


   package implementation backend    device  draws  seed  reps  \
0  Biogeme    biogeme_gpu     GPU  Tesla T4    500   123     5   
1  Biogeme    biogeme_gpu     GPU  Tesla T4   1000   123     5   
2  Biogeme    biogeme_gpu     GPU  Tesla T4   2000   123     5   
3  Biogeme    biogeme_gpu     GPU  Tesla T4   5000   123     5   
4  Biogeme    biogeme_gpu     GPU  Tesla T4  10000   123     5   

   initial_loglik  final_loglik  mean_estimation_time_sec  \
0    -2253.781763  -1447.293806                 12.007095   
1    -2253.780516  -1443.806334                 12.253011   
2    -2253.780561  -1445.539489                 14.051573   
3    -2253.780395  -1445.568122                 16.514007   
4    -2253.780058  -1444.210830                 20.973856   

   sd_estimation_time_sec  min_estimation_time_sec  max_estimation_time_sec  \
0                0.142665                11.875989                12.173011   
1                0.134468                12.046990                12.397184   

In [8]:
# ============================================================
# 6. Package outputs
# ============================================================

import zipfile

zip_file = proj_root / "results" / "biogeme_gpu_results.zip"
env_file = Path("/content/biogeme334_environment.json")

if zip_file.exists():
    zip_file.unlink()

zip_files = (
    sorted(raw_results_dir.glob("biogeme_gpu*"))
    + [summary_file, session_info_file, env_file]
)

with zipfile.ZipFile(
    zip_file,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zipf:

    for path in zip_files:
        if not path.exists():
            continue

        if path.is_relative_to(proj_root):
            arcname = path.relative_to(proj_root).as_posix()
        else:
            arcname = path.name

        zipf.write(path, arcname)

print("\nSaved result zip to:")
print(zip_file)



Saved result zip to:
/content/results/biogeme_gpu_results.zip
